# Level 1 · Step 8b — Claim check of the teacher replies (Colab Pro L4)

**Why:** the trial fine-tune (notebook 04) learned the format, but it also invents plain claims such as
"we have a Fortuner available". Step 8 only checks links, numbers and times, so the same kind of claim is still
in the training answers. This notebook reads every teacher answer against the memory it was written from and marks
it `supported` or `unsupported`. Step 8 then removes the unsupported ones locally, **before** the full training.

## What this notebook is — and is not

- **IS:** resumable. Every shard (1,000 replies) goes to the private dataset `ankit-iiitdmj/mbg-claims-<export>`.
- **IS NOT:** training. Nothing is changed here; the result is only a list of verdicts.

## GPU

Pick **L4** (Runtime → Change runtime type). Expected: **about 40 minutes, ~4 units** for ~13,000 replies.
Disconnect the runtime when Cell 11 is done.

## First-shard health check

The run aborts after the first shard if more than 5% of the verdicts cannot be read, or if (almost) everything
or nothing is marked unsupported.

In [ ]:
# Cell 2 — setup: clone repo + install dependencies (defensive)
REPO_URL  = "https://github.com/ankit8453/chat-AI-model.git"   # private repo -> asks for a GitHub token once
REPO_PATH = "/content/chat-AI-model"

import os, subprocess, sys, getpass

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", url, REPO_PATH], capture_output=True, text=True)

if os.path.isdir(REPO_PATH) and not os.path.isfile(os.path.join(REPO_PATH, "README.md")):
    print(f"Removing partial clone at {REPO_PATH} ...")
    subprocess.run(["rm", "-rf", REPO_PATH], check=True)
if not os.path.isdir(REPO_PATH):
    r = _clone(REPO_URL)
    if r.returncode != 0:
        print("Public clone failed (private repo?). Paste a GitHub token with read access:")
        tok = getpass.getpass("GitHub token: ")
        r = _clone(REPO_URL.replace("https://", f"https://{tok}@"))
        if r.returncode != 0:
            raise SystemExit("git clone failed:\n" + r.stderr[-1500:])
        subprocess.run(["git", "-C", REPO_PATH, "remote", "set-url", "origin", REPO_URL], check=False)
else:
    # Already cloned in this runtime -> pull the latest code (fixes pushed since the last run).
    r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only"], capture_output=True, text=True)
    if r.returncode != 0:
        tok = getpass.getpass("GitHub token (to pull updates): ")
        r = subprocess.run(["git", "-C", REPO_PATH, "pull", "--ff-only",
                            REPO_URL.replace("https://", f"https://{tok}@"), "main"], capture_output=True, text=True)
    print("git pull:", (r.stdout or r.stderr).strip()[-300:])

print("code version:", subprocess.run(["git", "-C", REPO_PATH, "log", "-1", "--oneline"],
                                      capture_output=True, text=True).stdout.strip())
os.chdir(REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, "src"))
print("Repo root contents:", sorted(os.listdir(".")))

_pip_packages = ["huggingface_hub>=0.24", "transformers>=4.45", "accelerate>=0.33", "bitsandbytes>=0.43"]
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_pip_packages], capture_output=True, text=True)
if proc.returncode != 0:
    print("PIP STDERR (tail):\n" + proc.stderr[-3000:])
    raise SystemExit("pip install failed — see tail above.")

# vLLM is optional: much faster, but newer releases may not support the T4. If it fails we fall back.
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm"], capture_output=True, text=True)
print("vllm install:", "ok" if proc.returncode == 0 else "FAILED (will use transformers 4-bit fallback)")
print("setup ok")

In [ ]:
# Cell 3 — HF Hub login
from huggingface_hub import login, HfApi
login()  # Colab inline widget — paste your Write token

_whoami = HfApi().whoami()
assert _whoami["name"] == "ankit-iiitdmj", (
    f"HF Hub token belongs to {_whoami['name']!r}, expected 'ankit-iiitdmj'."
)
HF_NS = _whoami["name"]
print(f"HF Hub ok: user={HF_NS}")

In [ ]:
# Cell 4 — GPU check + checker config
import subprocess
subprocess.run(["nvidia-smi"], check=False)
print()

from judge.engine import answer_config
cfg = answer_config()
cfg["max_new_tokens"] = 120          # one short JSON answer
for k, v in cfg.items():
    print(f"  {k}: {v}")

## Configuration

- Export: **`messages_2026-10-31`** (change `EXPORT` in Cell 6 for later exports)
- Input: `ankit-iiitdmj/mbg-sft-<EXPORT>` train / validation / test (teacher answers only; hand-overs need no
  check) + `ankit-iiitdmj/mbg-chat-candidates` → `<EXPORT>/sft_items.jsonl` (each reply's memory text)
- Checker: Qwen2.5-14B-Instruct-AWQ on the L4, greedy, prompt in `src/sft/claims.py`
- Output: `ankit-iiitdmj/mbg-claims-<EXPORT>` → `shards/`, then `claims_all.jsonl` + `summary.json`

In [ ]:
# Cell 6 — load the teacher answers + their memory text from the private HF datasets
import json
from collections import Counter
from huggingface_hub import hf_hub_download

EXPORT = "messages_2026-10-31"

def _jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

source_of = {it["item_id"]: it["source"] for it in
             _jsonl(hf_hub_download(f"{HF_NS}/mbg-chat-candidates", f"{EXPORT}/sft_items.jsonl", repo_type="dataset"))}
todo = []
for split in ("train", "validation", "test"):
    for r in _jsonl(hf_hub_download(f"{HF_NS}/mbg-sft-{EXPORT}", f"{split}.jsonl", repo_type="dataset")):
        if r["kind"] == "answer" and r["source_kind"] == "teacher":
            todo.append({"item_id": r["item_id"], "topic": r["topic"], "source": source_of[r["item_id"]],
                         "chat": r["messages"][1:-1], "reply": r["messages"][-1]["content"]})

print(f"teacher answers to check: {len(todo):,}")
print("by topic:", dict(Counter(t["topic"] for t in todo).most_common()))

In [ ]:
# Cell 7 — resume state from HF Hub (shards already written in earlier sessions)
from judge.hub import JudgeShardManager

ckpt_mgr = JudgeShardManager(HF_NS, EXPORT, prefix="mbg-claims")
done_rows, next_shard = ckpt_mgr.try_load_done()
done_ids = {r["item_id"] for r in done_rows}
remaining = [t for t in todo if t["item_id"] not in done_ids]

print(f"repo: {ckpt_mgr.repo_id} (private)")
print(f"already checked: {len(done_rows):,} in {next_shard} shards")
print(f"remaining:       {len(remaining):,}")
if not remaining:
    print("Nothing left — go to Cell 11.")

In [ ]:
# Cell 8 — build the checker engine (vLLM, else transformers 4-bit)
from judge.engine import build_engine
engine = build_engine(cfg, backend="auto")

In [ ]:
# Cell 9 — check loop: one shard at a time, pushed to HF before the next; first-shard health check
import time, datetime
from sft.claims import claim_messages, parse_claim

SHARD = cfg["shard_size"]
MAX_SHARDS_THIS_SESSION = None   # e.g. 2 to stop early on purpose; None = run until done / disconnected

def health_check(rows):
    n = len(rows)
    bad = sum(1 for r in rows if r["verdict"] is None) / n
    uns = sum(1 for r in rows if r["verdict"] == "unsupported") / n
    print(f"health: unreadable={bad:.1%}  unsupported={uns:.0%}  (n={n})")
    if bad > 0.05:
        raise SystemExit(f"ABORT: {bad:.0%} of the verdicts are not valid JSON — send this output to Claude.")
    if not (0.03 <= uns <= 0.70):
        raise SystemExit(f"ABORT: {uns:.0%} marked unsupported — the checker prompt is off, send this output to Claude.")
    print("health check passed")

t_start = time.time(); written_now = 0; shard_idx = next_shard
for s, i in enumerate(range(0, len(remaining), SHARD)):
    if MAX_SHARDS_THIS_SESSION is not None and s >= MAX_SHARDS_THIS_SESSION:
        print("Reached MAX_SHARDS_THIS_SESSION — stopping cleanly."); break
    batch = remaining[i:i + SHARD]
    t0 = time.time()
    convs = [claim_messages(t["source"], t["chat"], t["reply"], engine.count_tokens, engine.max_input_tokens) for t in batch]
    outputs = engine.generate_safe(convs)   # never crashes the shard
    rows = [{"item_id": t["item_id"], **parse_claim(out), "raw": (out or "")[:300], "model": engine.model_id,
             "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat()}
            for t, out in zip(batch, outputs)]
    dest = ckpt_mgr.save_shard(shard_idx, rows)
    if s == 0:
        health_check(rows)
    written_now += len(rows); shard_idx += 1
    rate = written_now / max(time.time() - t_start, 1e-6)
    left = len(remaining) - (i + len(batch))
    uns = sum(1 for r in rows if r["verdict"] == "unsupported") / len(rows)
    print(f"[shard {shard_idx-1}] {len(rows)} checked in {time.time()-t0:.0f}s | unsupported {uns:.0%} | "
          f"{rate:.1f} items/s | left {left:,} (~{left/max(rate,1e-6)/60:.0f} min) | pushed {dest}")

print(f"\nThis session checked {written_now:,} replies.")

## Summary + verdict

Re-loads **all** shards from the Hub, writes `claims_all.jsonl` and `summary.json` to
`ankit-iiitdmj/mbg-claims-<EXPORT>`, prints 8 flagged replies, and gives the verdict.

In [ ]:
# Cell 11 — aggregate all shards, push claims_all.jsonl + summary.json, show examples, verdict
import random, tempfile, os

all_rows, _ = ckpt_mgr.try_load_done()
rows = list({r["item_id"]: r for r in all_rows}.values())
by_id = {t["item_id"]: t for t in todo}
flagged = [r for r in rows if r["verdict"] == "unsupported"]

summary = {
    "export": EXPORT, "teacher_answers": len(todo), "checked": len(rows), "unsupported": len(flagged),
    "supported": sum(1 for r in rows if r["verdict"] == "supported"),
    "unreadable": sum(1 for r in rows if r["verdict"] is None),
    "unsupported_by_topic": dict(Counter(by_id[r["item_id"]]["topic"] for r in flagged if r["item_id"] in by_id).most_common()),
    "checker": rows[0]["model"] if rows else None,
}
tmp = tempfile.mkdtemp()
with open(os.path.join(tmp, "claims_all.jsonl"), "w", encoding="utf-8") as fh:
    for r in rows:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
with open(os.path.join(tmp, "summary.json"), "w", encoding="utf-8") as fh:
    json.dump(summary, fh, indent=2, ensure_ascii=False)
ckpt_mgr.save_file(os.path.join(tmp, "claims_all.jsonl"), "claims_all.jsonl", "all claim verdicts")
ckpt_mgr.save_file(os.path.join(tmp, "summary.json"), "summary.json", "claims summary")
print(json.dumps(summary, indent=2, ensure_ascii=False))

random.seed(0)
for r in random.sample(flagged, min(8, len(flagged))):
    t = by_id[r["item_id"]]
    print(f"\n[{t['topic']}] REPLY: {t['reply'][:220]!r}\n  UNSUPPORTED: {r['claim']!r}")

done = len(rows) >= len(todo)
print("\nProgress:", f"{len(rows):,}/{len(todo):,} checked", "(complete)" if done else "(run Cell 9 again)")
print("VERDICT:", "READY — Claude reviews a sample and rebuilds the dataset" if done else "NOT DONE YET — run Cell 9 again")

## Decision tree

- **READY** — send the Cell 11 output to Claude. Claude reviews a sample of the verdicts, removes the unsupported
  answers in step 8 and pushes the cleaned dataset. Then notebook 04 with `MODE = "full"`.
- **NOT DONE YET** — the session ended early. Run all again: Cell 7 resumes from the Hub.
- **ABORT in health check** — send the output to Claude; the prompt in `src/sft/claims.py` gets fixed.